# BioVault: Privacy-Preserving Federated Threat Detection for IoMT
### Clinical Threat Detection Benchmark on CICIoMT2024 with Opacus DP-SGD

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![Flower](https://img.shields.io/badge/FL-Flower-orange.svg)](https://flower.ai/)
[![Opacus](https://img.shields.io/badge/DP-Opacus-green.svg)](https://opacus.ai/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**BioVault** demonstrates collaborative IoMT threat intelligence across heterogeneous hospital fleets without centralizing Protected Health Information (PHI).

---
### Dataset Download Notice
> **Important**: In compliance with dataset redistribution terms, this notebook does not redistribute the raw CICIoMT2024 telemetry data.
> Please download the official dataset directly from the **Canadian Institute for Cybersecurity (CIC)**:
> [CICIoMT2024 Official Dataset Portal](https://www.unb.ca/cic/datasets/iomt-dataset-2024.html)
> Place the CSV files in `data/CICIoMT2024/`. If the dataset is not present, this notebook falls back to sample telemetry flows.


In [ ]:
import os
import sys
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from contracts import AttackClass, DPConfig
from data import load_ciomt, train_test_split, partition_non_iid, get_client_loaders, engineer_features, DEVICE_FLEET_NAMES
from model import ThreatMLP, compute_class_weights
from metrics import evaluate, confusion_matrix
from dp import make_client_private, get_privacy_spent
from opacus.validators import ModuleValidator

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())


## 1. Dataset Ingestion & Clinical Attack Taxonomy Mapping
Load network flow telemetry from the CICIoMT2024 dataset and map attack traffic to the 5-class clinical threat taxonomy:
- `BENIGN (0)`: Baseline clinical telemetry
- `DDOS (1)`: Volumetric floods targeting medical gateways
- `RANSOMWARE (2)`: Endpoint encryption payloads
- `DATA_TAMPERING (3)`: Vital sign and dosing integrity attacks
- `OTHER (4)`: Reconnaissance and port scans


In [ ]:
data_path = os.path.join('..', 'data', 'CICIoMT2024')
if not os.path.exists(data_path):
    data_path = os.path.join('..', 'samples', 'mixed_stream.csv')

print(f"Loading telemetry from: {data_path}")
df_clean = load_ciomt(data_path)
print(f"Loaded {len(df_clean)} flows with {len(df_clean.columns)} features.")
print("\nClass Distribution:")
print(df_clean['Label'].value_counts())


## 2. Non-IID Hospital Fleet Partitioning
We partition the client training pool into 5 non-IID hospital device fleets using Dirichlet label skew (alpha=0.5) and fleet-specific attack eligibility:
1. Infusion Pumps (Bedside Drug Delivery)
2. Patient Monitors (Vital Signs Telemetry)
3. Imaging Systems (PACS / Radiology)
4. Connected Ventilators (ICU Respiratory Support)
5. Clinical Lab Gateways (Diagnostic Infrastructure)


In [ ]:
train_pool, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, shuffle=True, stratify=df_clean['Label']
)

partitions = partition_non_iid(
    train_pool, num_clients=5, alpha=0.5, seed=42, stratify_by_device=True
)

fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
for i, (p, ax) in enumerate(zip(partitions, axes)):
    counts = p['Label'].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=['#2ca02c', '#1f77b4', '#d62728', '#ff7f0e', '#9467bd'])
    ax.set_title(f"Client {i}\n{DEVICE_FLEET_NAMES[i][:15]}...", fontsize=9)
    ax.set_xlabel("Attack Class")
    ax.set_xticks(range(5))
axes[0].set_ylabel("Flow Count")
plt.suptitle("Non-IID Device Fleet Label Distribution (Dirichlet alpha=0.5)", fontsize=12, y=1.05)
plt.tight_layout()
plt.show()


## 3. Short 5-Round Federated Learning Benchmark
We run 5 rounds of federated aggregation under two conditions:
1. **Standard Baseline FedAvg** (epsilon = infinity, Non-Private)
2. **BioVault DP-SGD FedAvg** (epsilon = 3.0, delta = 1e-5, Sample-Level DP-SGD)


In [ ]:
test_X, test_y, _ = engineer_features(test_df, fit_scaler=True)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(test_X).float(), torch.from_numpy(test_y).long()),
    batch_size=64,
    shuffle=False
)

client_loaders = [get_client_loaders(p, batch_size=64, seed=42) for p in partitions]
input_dim = test_X.shape[1]

def run_short_benchmark(use_dp: bool, target_epsilon: float = 3.0, num_rounds: int = 5):
    torch.manual_seed(42)
    np.random.seed(42)
    
    global_model = ThreatMLP(input_dim=input_dim, num_classes=5)
    if use_dp:
        global_model = ModuleValidator.fix(global_model)
        dp_cfg = DPConfig(target_epsilon=target_epsilon, epochs=1, batch_size=64)
    
    lr = 3e-4 if use_dp else 1e-3
    history = []
    
    for r in range(1, num_rounds + 1):
        sampled_clients = [0, 1, 3] if len(partitions) >= 3 else list(range(len(partitions)))
        client_updates = []
        eps_spent_round = 0.0
        
        for cid in sampled_clients:
            t_ldr, _ = client_loaders[cid]
            m = copy.deepcopy(global_model)
            cw = compute_class_weights(t_ldr, num_classes=5)
            opt = torch.optim.Adam(m.parameters(), lr=lr)
            
            if use_dp:
                p_m, p_opt, p_ldr, pe = make_client_private(m, opt, t_ldr, dp_cfg)
                crit = nn.CrossEntropyLoss(weight=cw)
                p_m.train()
                for x_b, y_b in p_ldr:
                    p_opt.zero_grad()
                    loss = crit(p_m(x_b), y_b)
                    loss.backward()
                    p_opt.step()
                eps_spent_round = get_privacy_spent(pe, delta=dp_cfg.target_delta)
                client_updates.append(copy.deepcopy(p_m._module.state_dict()))
            else:
                crit = nn.CrossEntropyLoss(weight=cw)
                m.train()
                for x_b, y_b in t_ldr:
                    opt.zero_grad()
                    loss = crit(m(x_b), y_b)
                    loss.backward()
                    opt.step()
                client_updates.append(copy.deepcopy(m.state_dict()))
                
        # FedAvg Aggregation
        avg_sd = {}
        for k in client_updates[0].keys():
            avg_sd[k] = torch.stack([u[k].float() for u in client_updates]).mean(dim=0)
        global_model.load_state_dict(avg_sd)
        
        # Central Evaluation
        res = evaluate(global_model, test_loader, num_classes=5)
        history.append({
            'round': r,
            'loss': res['loss'],
            'accuracy': res['accuracy'],
            'macro_f1': res['macro_f1'],
            'ransomware_recall': res['ransomware_recall'],
            'epsilon_spent': eps_spent_round
        })
        print(f"Round {r:2d} | Loss: {res['loss']:.4f} | Acc: {res['accuracy']:.4f} | F1: {res['macro_f1']:.4f} | Recall: {res['ransomware_recall']:.4f}")
        
    return pd.DataFrame(history)

print("=== Running Non-Private Baseline FL ===")
baseline_history = run_short_benchmark(use_dp=False, num_rounds=5)

print("\n=== Running BioVault DP-SGD FL (Epsilon=3.0) ===")
dp_history = run_short_benchmark(use_dp=True, target_epsilon=3.0, num_rounds=5)


## 4. Benchmark Performance & Trade-Off Visualization
Compare convergence trajectories across rounds for Macro-F1 and Ransomware Recall.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(baseline_history['round'], baseline_history['macro_f1'], 'o-', label='Baseline (Non-Private)', color='#1f77b4', lw=2)
ax1.plot(dp_history['round'], dp_history['macro_f1'], 's--', label='BioVault DP-SGD (epsilon=3.0)', color='#ff7f0e', lw=2)
ax1.set_title("Macro-F1 Score Trajectory", fontsize=11, fontweight='bold')
ax1.set_xlabel("Federated Communication Round")
ax1.set_ylabel("Macro-F1 Score")
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(baseline_history['round'], baseline_history['ransomware_recall'], 'o-', label='Baseline (Non-Private)', color='#2ca02c', lw=2)
ax2.plot(dp_history['round'], dp_history['ransomware_recall'], 's--', label='BioVault DP-SGD (epsilon=3.0)', color='#d62728', lw=2)
ax2.set_title("Ransomware Detection Recall Trajectory", fontsize=11, fontweight='bold')
ax2.set_xlabel("Federated Communication Round")
ax2.set_ylabel("Recall Rate")
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.suptitle("BioVault Federated Threat Detection: Privacy vs. Utility", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

report_df = pd.DataFrame([
    {
        'Strategy': 'Non-Private Baseline (FedAvg)',
        'Privacy Guarantee (epsilon, delta)': 'None (epsilon = inf)',
        'Final Macro-F1': f"{baseline_history['macro_f1'].iloc[-1]:.4f}",
        'Final Ransomware Recall': f"{baseline_history['ransomware_recall'].iloc[-1]:.4f}",
        'Final Accuracy': f"{baseline_history['accuracy'].iloc[-1]:.4f}",
    },
    {
        'Strategy': 'BioVault DP-SGD (Flower + Opacus)',
        'Privacy Guarantee (epsilon, delta)': 'epsilon = 3.0, delta = 1e-5',
        'Final Macro-F1': f"{dp_history['macro_f1'].iloc[-1]:.4f}",
        'Final Ransomware Recall': f"{dp_history['ransomware_recall'].iloc[-1]:.4f}",
        'Final Accuracy': f"{dp_history['accuracy'].iloc[-1]:.4f}",
    }
])

print("\nBENCHMARK RESULTS SUMMARY:")
print(report_df.to_string(index=False))
